# AMPS I · Week 15 — HW14: the lid-driven cavity at Re = 100, 400, 1000
**Due:** before the Week 16 session · **Points:** 100 · **Submit:** this notebook (`.ipynb`) with all outputs

**Rules**
- Before submitting: *Runtime → Restart session and run all*. Notebooks that do not run end to end receive 0.
- An error inside a `%%fenicsx` cell does **not** stop *Run all*. Read every output before you submit.
- LLM assistants are encouraged. Fill in the **AI-usage log** at the end.
- Every number you report must come from a cell you ran. Print the SNES converged reason of every solve.
- **Mesh for Q2–Q3:** `diagonal=mesh.DiagonalType.right` — not the lecture's `crossed` mesh.

| Question | Topic | Points |
|---|---|:---:|
| Q1 | Newton for NS, continuation, PCD, H(div) | 20 |
| Q2 | The cavity vs Ghia et al. on three meshes | 40 |
| Q3 | GMRES iterations: mass matrix vs PCD along a continuation path | 20 |
| Q4 | LLM debugging: a cavity script with four bugs | 20 |

## ⚙️ Setup — run once per runtime
Installs DOLFINx 0.11 and registers the `%%fenicsx` cell magic. Re-run after every runtime restart.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  FEniCSx on Google Colab — run once per runtime (≈ 3–5 min)
#  https://github.com/SeoulTechPSE/fenicsx-colab
# ══════════════════════════════════════════════════════════════════
from google.colab import drive
import os, multiprocessing, subprocess
from pathlib import Path

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/seoultechpse/fenicsx-colab.git'
REPO_DIR = Path('/content/fenicsx-colab')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

# OpenMPI 5.x slot patch: Colab has 2 CPUs; allow up to 4 ranks (oversubscribed)
N_PROC = 4
with open('/root/hostfile', 'w') as f:
    f.write(f'localhost slots={N_PROC}\n')
os.environ['OMPI_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['PRTE_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['OMPI_MCA_osc'] = '^ucx'   # skip the UCX one-sided module (harmless start-up message)

DOLFINX_VERSION = '0.11'
USE_COMPLEX = False
USE_CLEAN   = False
opts = [f'--version {DOLFINX_VERSION}']
if USE_COMPLEX: opts.append('--complex')
if USE_CLEAN:   opts.append('--clean')
get_ipython().run_line_magic('run', f"{REPO_DIR / 'setup_fenicsx.py'} {' '.join(opts)}")

# Folder shared by %%fenicsx cells (computation) and ordinary cells (plotting)
os.makedirs('/tmp/fnx', exist_ok=True)

In [ ]:
%%fenicsx --info -np 2

## Q1 — Theory (20 pts)

**(a) (6)** Let $c(\mathbf w;\mathbf u,\mathbf v)=((\mathbf w\cdot\nabla)\mathbf u,\mathbf v)$. Compute the Gâteaux derivative of $\mathbf u\mapsto c(\mathbf u;\mathbf u,\mathbf v)$ in the direction $\delta\mathbf u$.
Which term does the Picard (Oseen) iteration drop? State the convergence order of each and the price of each iteration.

**(b) (4)** For small data ($\nu^2>\mathscr N\|\mathbf f\|_{-1}$, $\mathscr N$ the continuity constant of $c$) the steady NS solution is unique and Picard converges from any start. Why does that not help at
Re = 1000, and why does continuation in Re help Newton?

**(c) (6) PCD.** With $F=\nu A+N(\mathbf w)$ the convection–diffusion operator on the velocity and $F_p=\nu A_p+N_p(\mathbf w)$ "the same" operator on the pressure space, assume the
commutator $FM_u^{-1}B^\top-B^\top M_p^{-1}F_p$ is small. Derive $S=BF^{-1}B^\top\approx A_pF_p^{-1}M_p$ (using $BM_u^{-1}B^\top\approx A_p$), hence $S^{-1}\approx M_p^{-1}F_pA_p^{-1}$. What does
$M_p/\nu$ miss?

**(d) (4) H(div).** Show: if $\nabla\cdot V_h\subseteq Q_h$ (e.g. RT$_{k+1}$/DG$_k$), the discrete incompressibility constraint $(\nabla\cdot\mathbf u_h,q_h)=0$ for all $q_h\in Q_h$ implies
$\nabla\cdot\mathbf u_h=0$ pointwise. Why is that not true for Taylor–Hood?

> *your answers to (a)–(d)*

## Q2 — The cavity vs Ghia et al. on three meshes (40 pts)

**(a) (15)** `right` mesh, $N=32,64,128$, continuation Re = 100 → 400 → 1000 (Newton + MUMPS). For each $N$ and Re: SNES reason, Newton iterations, $\max|u-u_{\text{Ghia}}|$ along
$x=0.5$ and $\max|v-v_{\text{Ghia}}|$ along $y=0.5$ — once over all of Ghia's points and once without $x=0.9063$. Save the $N=128$ profiles.

**(b) (10)** Plot $u(0.5,y)$ and $v(x,0.5)$ for the three Re ($N=128$) with Ghia's points.

**(c) (15)** How do the errors behave under refinement — what limits them at Re = 100? Look at Ghia's value $v(0.9063,0.5)=-0.23827$ at Re = 400: is it consistent with your
results and with its neighbours? Compare the `right` mesh with the lecture's `crossed` numbers.

In [ ]:
%%fenicsx -np 2
# TODO: Q2(a) the cavity on the right mesh, N = 32, 64, 128

from mpi4py import MPI
from petsc4py import PETSc
import time
import numpy as np
import ufl
import basix.ufl
import dolfinx.fem.petsc
from dolfinx import fem, mesh, geometry
from dolfinx.fem.petsc import LinearProblem, NonlinearProblem

comm = MPI.COMM_WORLD

def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)
# Ghia, Ghia & Shin (1982), Tables I and II (transcription: gist ivan-pi 3e9326d1..., caa6c673...)
GHIA_Y = np.array([1.0000, 0.9766, 0.9688, 0.9609, 0.9531, 0.8516, 0.7344, 0.6172, 0.5000, 0.4531, 0.2813, 0.1719, 0.1016, 0.0703, 0.0625, 0.0547, 0.0000])
GHIA_U = {100: [1.0, 0.84123, 0.78871, 0.73722, 0.68717, 0.23151, 0.00332, -0.13641, -0.20581, -0.21090, -0.15662, -0.10150, -0.06434, -0.04775, -0.04192, -0.03717, 0.0],
          400: [1.0, 0.75837, 0.68439, 0.61756, 0.55892, 0.29093, 0.16256, 0.02135, -0.11477, -0.17119, -0.32726, -0.24299, -0.14612, -0.10338, -0.09266, -0.08186, 0.0],
          1000: [1.0, 0.65928, 0.57492, 0.51117, 0.46604, 0.33304, 0.18719, 0.05702, -0.06080, -0.10648, -0.27805, -0.38289, -0.29730, -0.22220, -0.20196, -0.18109, 0.0]}
GHIA_X = np.array([1.0000, 0.9688, 0.9609, 0.9531, 0.9453, 0.9063, 0.8594, 0.8047, 0.5000, 0.2344, 0.2266, 0.1563, 0.0938, 0.0781, 0.0703, 0.0625, 0.0000])
GHIA_V = {100: [0.0, -0.05906, -0.07391, -0.08864, -0.10313, -0.16914, -0.22445, -0.24533, 0.05454, 0.17527, 0.17507, 0.16077, 0.12317, 0.10890, 0.10091, 0.09233, 0.0],
          400: [0.0, -0.12146, -0.15663, -0.19254, -0.22847, -0.23827, -0.44993, -0.38598, 0.05186, 0.30174, 0.30203, 0.28124, 0.22965, 0.20920, 0.19713, 0.18360, 0.0],
          1000: [0.0, -0.21388, -0.27669, -0.33714, -0.39188, -0.51500, -0.42665, -0.31966, 0.02526, 0.32235, 0.33075, 0.37095, 0.32627, 0.30353, 0.29012, 0.27485, 0.0]}

In [ ]:
# TODO: Q2(b) plots

> *your table and answers to (c)*

## Q3 — GMRES iterations along a continuation path (20 pts)

Use the lab's `cavity_krylov` (install `fenicsx-pctools` first) on the `right` mesh.

**(a) (12)** PCD: Re = 100 → 250 → 500 → 750 → 1000 on $N=32$ and $N=64$. Mass matrix: the same path on $N=32$ (it stops when GMRES fails). Table of Newton iterations,
GMRES iterations per Newton step and SNES reasons.

**(b) (8)** Plot GMRES per Newton step against Re. Is PCD mesh-independent? Re-independent? Why does the mass-matrix version fail, and what would make PCD better for this
enclosed flow (boundary conditions of $A_p$, $F_p$)?

In [ ]:
# fenicsx-pctools (PCD preconditioner) into the %%fenicsx environment on Colab. That environment has no pip: bootstrap it first.
# --no-deps keeps DOLFINx 0.11 untouched (the package declares dolfinx <= 0.11.0.dev0 but works with 0.11.0). Not on Colab: skipped.
import os, subprocess
PY = "/content/micromamba/envs/fenicsx/bin/python"
if os.path.exists(PY):
    run = lambda cmd: subprocess.run(cmd, capture_output=True, text=True)
    if run([PY, "-c", "import pip"]).returncode != 0:
        print("ensurepip:", "ok" if run([PY, "-m", "ensurepip", "--upgrade"]).returncode == 0 else "FAILED")
    r = run([PY, "-m", "pip", "install", "--no-deps", "-q", "git+https://gitlab.com/rafinex-external-rifle/fenicsx-pctools.git"])
    print("fenicsx-pctools:", "ok" if r.returncode == 0 else "FAILED\n" + r.stderr[-1500:])
else:
    print("not on Colab: install fenicsx-pctools yourself (pip install --no-deps git+https://gitlab.com/rafinex-external-rifle/fenicsx-pctools.git)")

In [ ]:
%%fenicsx -np 2
# TODO: Q3(a) mass vs PCD along Re = 100, 250, 500, 750, 1000 (right mesh); save the table to /tmp/fnx/hw14_q3.npy

from mpi4py import MPI
from petsc4py import PETSc
import time
import numpy as np
import ufl
import basix.ufl
import dolfinx.fem.petsc
from dolfinx import fem, mesh, geometry
from dolfinx.fem.petsc import LinearProblem, NonlinearProblem

comm = MPI.COMM_WORLD

def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)
# Ghia, Ghia & Shin (1982), Tables I and II (transcription: gist ivan-pi 3e9326d1..., caa6c673...)
GHIA_Y = np.array([1.0000, 0.9766, 0.9688, 0.9609, 0.9531, 0.8516, 0.7344, 0.6172, 0.5000, 0.4531, 0.2813, 0.1719, 0.1016, 0.0703, 0.0625, 0.0547, 0.0000])
GHIA_U = {100: [1.0, 0.84123, 0.78871, 0.73722, 0.68717, 0.23151, 0.00332, -0.13641, -0.20581, -0.21090, -0.15662, -0.10150, -0.06434, -0.04775, -0.04192, -0.03717, 0.0],
          400: [1.0, 0.75837, 0.68439, 0.61756, 0.55892, 0.29093, 0.16256, 0.02135, -0.11477, -0.17119, -0.32726, -0.24299, -0.14612, -0.10338, -0.09266, -0.08186, 0.0],
          1000: [1.0, 0.65928, 0.57492, 0.51117, 0.46604, 0.33304, 0.18719, 0.05702, -0.06080, -0.10648, -0.27805, -0.38289, -0.29730, -0.22220, -0.20196, -0.18109, 0.0]}
GHIA_X = np.array([1.0000, 0.9688, 0.9609, 0.9531, 0.9453, 0.9063, 0.8594, 0.8047, 0.5000, 0.2344, 0.2266, 0.1563, 0.0938, 0.0781, 0.0703, 0.0625, 0.0000])
GHIA_V = {100: [0.0, -0.05906, -0.07391, -0.08864, -0.10313, -0.16914, -0.22445, -0.24533, 0.05454, 0.17527, 0.17507, 0.16077, 0.12317, 0.10890, 0.10091, 0.09233, 0.0],
          400: [0.0, -0.12146, -0.15663, -0.19254, -0.22847, -0.23827, -0.44993, -0.38598, 0.05186, 0.30174, 0.30203, 0.28124, 0.22965, 0.20920, 0.19713, 0.18360, 0.0],
          1000: [0.0, -0.21388, -0.27669, -0.33714, -0.39188, -0.51500, -0.42665, -0.31966, 0.02526, 0.32235, 0.33075, 0.37095, 0.32627, 0.30353, 0.29012, 0.27485, 0.0]}

In [ ]:
# TODO: Q3(b) plot

> *your table and answers to (b)*

## Q4 — LLM debugging: a cavity script with four bugs (20 pts)

An LLM was asked: *"DOLFINx 0.11, -np 2: steady lid-driven cavity, Taylor-Hood, Newton with Reynolds-number continuation 100 -> 400 -> 1000 on a 32 x 32 mesh;
print the maximal deviation from Ghia et al. along both centre lines."* It produced the cell below (run it as given).

**(a) (12)** Find the four bugs: line, what is wrong, how it shows in the output, how you found it. They hide each other — fix them one at a time.

**(b) (5)** Corrected cell: Newton iterations and deviations for the three Re (compare with the lab, `crossed` mesh, or Q2).

**(c) (3)** Which bug would a reader of the printed table never notice?

In [ ]:
%%fenicsx -np 2
from mpi4py import MPI
from petsc4py import PETSc
import time
import numpy as np
import ufl
import basix.ufl
import dolfinx.fem.petsc
from dolfinx import fem, mesh, geometry
from dolfinx.fem.petsc import LinearProblem, NonlinearProblem

comm = MPI.COMM_WORLD

def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)
# Ghia, Ghia & Shin (1982), Tables I and II (transcription: gist ivan-pi 3e9326d1..., caa6c673...)
GHIA_Y = np.array([1.0000, 0.9766, 0.9688, 0.9609, 0.9531, 0.8516, 0.7344, 0.6172, 0.5000, 0.4531, 0.2813, 0.1719, 0.1016, 0.0703, 0.0625, 0.0547, 0.0000])
GHIA_U = {100: [1.0, 0.84123, 0.78871, 0.73722, 0.68717, 0.23151, 0.00332, -0.13641, -0.20581, -0.21090, -0.15662, -0.10150, -0.06434, -0.04775, -0.04192, -0.03717, 0.0],
          400: [1.0, 0.75837, 0.68439, 0.61756, 0.55892, 0.29093, 0.16256, 0.02135, -0.11477, -0.17119, -0.32726, -0.24299, -0.14612, -0.10338, -0.09266, -0.08186, 0.0],
          1000: [1.0, 0.65928, 0.57492, 0.51117, 0.46604, 0.33304, 0.18719, 0.05702, -0.06080, -0.10648, -0.27805, -0.38289, -0.29730, -0.22220, -0.20196, -0.18109, 0.0]}
GHIA_X = np.array([1.0000, 0.9688, 0.9609, 0.9531, 0.9453, 0.9063, 0.8594, 0.8047, 0.5000, 0.2344, 0.2266, 0.1563, 0.0938, 0.0781, 0.0703, 0.0625, 0.0000])
GHIA_V = {100: [0.0, -0.05906, -0.07391, -0.08864, -0.10313, -0.16914, -0.22445, -0.24533, 0.05454, 0.17527, 0.17507, 0.16077, 0.12317, 0.10890, 0.10091, 0.09233, 0.0],
          400: [0.0, -0.12146, -0.15663, -0.19254, -0.22847, -0.23827, -0.44993, -0.38598, 0.05186, 0.30174, 0.30203, 0.28124, 0.22965, 0.20920, 0.19713, 0.18360, 0.0],
          1000: [0.0, -0.21388, -0.27669, -0.33714, -0.39188, -0.51500, -0.42665, -0.31966, 0.02526, 0.32235, 0.33075, 0.37095, 0.32627, 0.30353, 0.29012, 0.27485, 0.0]}

msh = mesh.create_unit_square(comm, 32, 32, diagonal=mesh.DiagonalType.crossed)
TH = basix.ufl.mixed_element([basix.ufl.element("Lagrange", "triangle", 2, shape=(2,)), basix.ufl.element("Lagrange", "triangle", 1)])
W = fem.functionspace(msh, TH)
W0, _ = W.sub(0).collapse(); W1, _ = W.sub(1).collapse()
lid = fem.Function(W0)
lid.interpolate(lambda x: np.vstack([np.isclose(x[1], 1.0).astype(float), np.zeros(x.shape[1])]))
msh.topology.create_connectivity(1, 2)
bf = mesh.exterior_facet_indices(msh.topology)
bcs = [fem.dirichletbc(lid, fem.locate_dofs_topological((W.sub(0), W0), 1, bf), W.sub(0)),
       fem.dirichletbc(fem.Function(W1), fem.locate_dofs_geometrical((W.sub(1), W1), lambda x: np.isclose(x[0], 0) & np.isclose(x[1], 0)), W.sub(1))]

Re = 100.0
w = fem.Function(W)
u, p = ufl.split(w); v, q = ufl.TestFunctions(W)
F = (1 / Re * ufl.inner(ufl.grad(u), ufl.grad(v)) + ufl.inner(ufl.dot(ufl.grad(u), u), v) - p * ufl.div(v) - q * ufl.div(u)) * ufl.dx
problem = NonlinearProblem(F, w, bcs=bcs, petsc_options_prefix="cavity_",
                           petsc_options={"snes_type": "newtonls", "snes_rtol": 1e-10, "snes_max_it": 3,
                                          "ksp_type": "preonly", "pc_type": "lu", "pc_factor_mat_solver_type": "mumps"})

tree = geometry.bb_tree(msh, 2)
def sample(points, comp):
    uh = w.sub(0).collapse()
    P = np.c_[points, np.zeros(len(points))]
    cells = geometry.compute_colliding_cells(msh, geometry.compute_collisions_points(tree, P), P)
    out = np.full(len(P), -np.inf)
    for i in range(len(P)):
        c = cells.links(i)
        if len(c):
            out[i] = uh.eval(P[i], c[:1])[comp]
    res = np.empty_like(out); comm.Allreduce(out, res, op=MPI.MAX)
    return res

for Re in (100, 400, 1000):
    w.x.array[:] = 0.0                                   # clean start for every Reynolds number
    problem.solve()
    du = np.abs(sample(np.c_[np.full(17, 0.5), GHIA_Y], 0) - GHIA_U[Re]).max()
    dv = np.abs(sample(np.c_[np.full(17, 0.5), GHIA_X], 1) - GHIA_V[Re]).max()
    say(f"Re = {Re:4d}: {problem.solver.getIterationNumber()} Newton iterations, max|u - Ghia| = {du:.4f}, max|v - Ghia| = {dv:.4f}")

In [ ]:
%%fenicsx -np 2
# TODO: Q4 your diagnostic copy and the corrected cell

> *your table for (a) and answers to (b), (c)*

## 🤖 AI-usage log

*Required in every assignment and project. Using an LLM is encouraged and never penalized — an empty or vague log is.*

| # | Tool / model | What I asked (key prompt, shortened) | What it got wrong or missed | How I detected it | How I fixed it |
|---|---|---|---|---|---|
| 1 | | | | | |
| 2 | | | | | |
| 3 | | | | | |

**Code provenance** — which cells/functions were mostly generated, which did you write or substantially change?

> …

**Verification** — how do you know the final result is correct (MMS rates, benchmark value, conservation check, parallel = serial)?

> …

**Reflection (2–3 sentences)** — what would you do differently when prompting next time?

> …